# A Practical Guide to ROC-AUC, PR-AUC, and R-squared

## Goal of the lesson

In this lesson we will study three important metrics for evaluating machine learning models:
1. **ROC-AUC** - for evaluating the quality of binary classification
2. **PR-AUC** - for evaluating classification under class imbalance
3. **R-squared** - for evaluating regression quality


## Importing the required libraries


In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    r2_score, mean_squared_error, classification_report, confusion_matrix
)


In [ ]:
# Plot style configuration
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


## 1. ROC-AUC: Evaluating Binary Classification Quality


In [ ]:
# Load the breast cancer dataset (a classic binary classification task)
cancer_data = load_breast_cancer()
X_cancer = cancer_data.data
y_cancer = cancer_data.target

print(f"Dataset size: {X_cancer.shape}")
print(f"Classes: {cancer_data.target_names}")
print(f"Class distribution: {np.bincount(y_cancer)}")


In [ ]:
# Split the data into train and test sets
X_train_cancer, X_test_cancer, y_train_cancer, y_test_cancer = train_test_split(
    X_cancer, y_cancer, test_size=0.3, random_state=42, stratify=y_cancer
)


In [ ]:
# Logistic regression
lr_model = LogisticRegression(random_state=42, max_iter=10000)
lr_model.fit(X_train_cancer, y_train_cancer)

# Random forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_cancer, y_train_cancer)


In [ ]:
# Get predicted probabilities (needed to build the ROC curve, since ROC analyzes how well a score ranks cases across thresholds)
# predict_proba(X) returns an array of shape (n_samples, n_classes) with per-class probabilities.
# [:, 1] - take the probability of the "positive" class (class index 1). For a binary task this matters: ROC is built on the positive-class score.
lr_proba = lr_model.predict_proba(X_test_cancer)[:, 1]
rf_proba = rf_model.predict_proba(X_test_cancer)[:, 1]

# Compute ROC curves
# roc_curve(y_true, y_score) returns:
#  - fpr (False Positive Rate) - fraction of false positives among all negatives,
#  - tpr (True Positive Rate, sensitivity/Recall) - fraction of true positives among all positives,
#  - thresholds - list of thresholds used to compute the (fpr, tpr) pairs.
# Note: thresholds are sorted in decreasing order of score; when the threshold -> -inf all objects are positive (TPR=1, FPR=1); when threshold -> +inf all are negative (TPR=0, FPR=0).
lr_fpr, lr_tpr, lr_thresholds = roc_curve(y_test_cancer, lr_proba)
rf_fpr, rf_tpr, rf_thresholds = roc_curve(y_test_cancer, rf_proba)

# Compute the area under the ROC curve (AUC).
lr_auc = auc(lr_fpr, lr_tpr)
rf_auc = auc(rf_fpr, rf_tpr)

# Plot ROC curves to compare the two models.
plt.figure(figsize=(15, 5))

# First subplot (of three): the curves themselves and the diagonal line of a random classifier.
plt.subplot(1, 3, 1)

# ROC lines for logistic regression and random forest; the legend shows AUC rounded to three decimals.
plt.plot(lr_fpr, lr_tpr, label=f'Logistic regression (AUC = {lr_auc:.3f})', linewidth=2)
plt.plot(rf_fpr, rf_tpr, label=f'Random forest (AUC = {rf_auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random prediction (AUC = 0.5)')

# For clarity, mark several characteristic probability thresholds on the logistic-regression ROC curve.
for thresh in [0.9, 0.5, 0.3, 0.1]:
    # Find the index of the closest threshold
    idx = np.argmin(np.abs(lr_thresholds - thresh))
    plt.scatter(lr_fpr[idx], lr_tpr[idx], marker='o', color='red')
    plt.text(lr_fpr[idx]+0.02, lr_tpr[idx]-0.02, f'{thresh:.1f}', fontsize=10)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC curves for the breast cancer classification task')
plt.legend()
plt.grid(True, alpha=0.3)

print(f"ROC-AUC for logistic regression: {lr_auc:.3f}")
print(f"ROC-AUC for random forest: {rf_auc:.3f}")


## 2. PR-AUC: Quality Analysis for Imbalanced Data


In [ ]:
# Create an imbalanced dataset for demonstration
X_imbalanced, y_imbalanced = make_classification(
    n_samples=1000, n_features=20, n_informative=10, n_redundant=10,
    n_clusters_per_class=1, weights=[0.9, 0.1], random_state=42
)

print(f"Class distribution in the imbalanced dataset: {np.bincount(y_imbalanced)}")
print(f"Percentage of positive examples: {y_imbalanced.mean():.1%}")


In [ ]:
# Split the data
X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imbalanced, y_imbalanced, test_size=0.3, random_state=42, stratify=y_imbalanced
)


In [ ]:
# Train the models
lr_imb = LogisticRegression(random_state=42)
lr_imb.fit(X_train_imb, y_train_imb)

rf_imb = RandomForestClassifier(n_estimators=100, random_state=42)
rf_imb.fit(X_train_imb, y_train_imb)


In [ ]:
# Get predicted probabilities
lr_imb_proba = lr_imb.predict_proba(X_test_imb)[:, 1]
rf_imb_proba = rf_imb.predict_proba(X_test_imb)[:, 1]

# Compute PR curves
lr_precision, lr_recall, lr_thresholds = precision_recall_curve(y_test_imb, lr_imb_proba)
rf_precision, rf_recall, rf_thresholds = precision_recall_curve(y_test_imb, rf_imb_proba)

# Compute PR-AUC
lr_pr_auc = average_precision_score(y_test_imb, lr_imb_proba)
rf_pr_auc = average_precision_score(y_test_imb, rf_imb_proba)

# Baseline for PR-AUC (proportion of positive examples)
baseline_pr = y_test_imb.mean()

# Plot PR curves
plt.subplot(1, 3, 2)
plt.plot(lr_recall, lr_precision, label=f'Logistic regression (PR-AUC = {lr_pr_auc:.3f})', linewidth=2)
plt.plot(rf_recall, rf_precision, label=f'Random forest (PR-AUC = {rf_pr_auc:.3f})', linewidth=2)
plt.axhline(y=baseline_pr, color='k', linestyle='--',
           label=f'Baseline (PR-AUC = {baseline_pr:.3f})')

# Add points for a few thresholds
for thresh in [0.9, 0.5, 0.1]:
    # Find the closest threshold index for logistic regression
    idx = np.argmin(np.abs(lr_thresholds - thresh))
    plt.scatter(lr_recall[idx], lr_precision[idx], marker='o', color='red')
    plt.text(lr_recall[idx] + 0.02, lr_precision[idx] - 0.02, f'{thresh:.1f}', fontsize=10, color='red')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('PR curves for the imbalanced dataset')
plt.legend()
plt.grid(True, alpha=0.3)

print(f"PR-AUC for logistic regression: {lr_pr_auc:.3f}")
print(f"PR-AUC for random forest: {rf_pr_auc:.3f}")
print(f"PR-AUC baseline: {baseline_pr:.3f}")


## 3. R-squared: Regression Quality Analysis


In [ ]:
# Load a ready-made dataset
from sklearn.preprocessing import LabelEncoder

df_dataset = pd.read_csv('airline_passenger_satisfaction.csv')
label_encoder = LabelEncoder()
columns = ['Gender', 'Type of Travel', 'Customer Type', 'Class']
for col in columns:
    df_dataset[col] = label_encoder.fit_transform(df_dataset[col])
df_dataset.drop(columns=['id', 'Unnamed: 0'], inplace=True)
df_dataset.dropna(inplace=True)

print(f"Dataset size after cleaning: {df_dataset.shape}")
print(f"Columns in the dataset: {list(df_dataset.columns)}")


In [ ]:
# R-squared requires a regression task, so we choose a numeric target variable.
# We will predict "Arrival Delay in Minutes" based on other factors
target_column = 'Arrival Delay in Minutes'
print(f"\n🎯 Regression target variable: {target_column}")

# Check the statistics of the target variable
print(f"Arrival delay statistics:")
print(f"  Mean: {df_dataset[target_column].mean():.2f} minutes")
print(f"  Median: {df_dataset[target_column].median():.2f} minutes")
print(f"  Standard deviation: {df_dataset[target_column].std():.2f} minutes")
print(f"  Min: {df_dataset[target_column].min():.2f} minutes")
print(f"  Max: {df_dataset[target_column].max():.2f} minutes")



In [ ]:
# Prepare the data for regression
X_airline = df_dataset.drop(columns=[target_column, 'satisfaction'])  # Drop the target variable and satisfaction
y_airline = df_dataset[target_column]

print(f"\nFeatures for prediction ({len(X_airline.columns)} columns):")
for i, col in enumerate(X_airline.columns):
    print(f"  {i+1:2d}. {col}")


In [ ]:
# Split the data into train and test sets
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_airline, y_airline, test_size=0.3, random_state=42
)

print(f"\nSample sizes:")
print(f"  Training: {X_train_reg.shape[0]} samples")
print(f"  Test: {X_test_reg.shape[0]} samples")

# Train regression models to predict arrival delay
print("\n🤖 Training models to predict arrival delay...")


In [ ]:
# Linear regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_reg, y_train_reg)

# Random forest regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train_reg, y_train_reg)



In [ ]:
# Make predictions
lin_pred = lin_reg.predict(X_test_reg)
rf_pred = rf_reg.predict(X_test_reg)

# Compute R-squared
lin_r2 = r2_score(y_test_reg, lin_pred)
rf_r2 = r2_score(y_test_reg, rf_pred)



In [ ]:
# Compute MAE and RMSE for additional information
from sklearn.metrics import mean_absolute_error
lin_mae = mean_absolute_error(y_test_reg, lin_pred)
rf_mae = mean_absolute_error(y_test_reg, rf_pred)
lin_rmse = np.sqrt(mean_squared_error(y_test_reg, lin_pred))
rf_rmse = np.sqrt(mean_squared_error(y_test_reg, rf_pred))

print(f"📈 Arrival delay prediction results:")
print(f"\nLinear regression:")
print(f"  - R² = {lin_r2:.4f} (explains {lin_r2*100:.2f}% of the variance)")
print(f"  - MAE = {lin_mae:.2f} minutes (mean absolute error)")
print(f"  - RMSE = {lin_rmse:.2f} minutes (root mean squared error)")

print(f"\nRandom forest:")
print(f"  - R² = {rf_r2:.4f} (explains {rf_r2*100:.2f}% of the variance)")
print(f"  - MAE = {rf_mae:.2f} minutes (mean absolute error)")
print(f"  - RMSE = {rf_rmse:.2f} minutes (root mean squared error)")


In [ ]:
# Interpretation of the results
print(f"\n✈️ Interpretation of the results for the airline:")

def interpret_r2(r2_value, model_name):
    if r2_value > 0.8:
        return f"• {model_name}: Excellent ability to predict delays"
    elif r2_value > 0.6:
        return f"• {model_name}: Good ability to predict delays"
    elif r2_value > 0.4:
        return f"• {model_name}: Moderate ability to predict delays"
    elif r2_value > 0.2:
        return f"• {model_name}: Weak ability to predict delays"
    else:
        return f"• {model_name}: Very weak model, no better than random prediction"

print(interpret_r2(lin_r2, "Linear regression"))
print(interpret_r2(rf_r2, "Random forest"))


In [ ]:
# Practical implications
print(f"\n💼 Practical applications:")
if max(lin_r2, rf_r2) > 0.6:
    print(f"• The models can be used for operations planning")
    print(f"• The accuracy is enough to inform passengers")
else:
    print(f"• The models need improvement before practical use")
    print(f"• Additional factors or different approaches are required")

# Feature importance analysis (for the random forest)
# This metric shows which features most strongly influence the prediction of the target variable.
print(f"\n🎯 Most important factors for predicting the delay:")

# rf_reg.feature_importances_ - an array of numbers (from 0 to 1),
# where each feature has its own "contribution" to the model.
feature_importance = rf_reg.feature_importances_

# X_airline.columns - names of the features (columns) the model was trained on.
feature_names = X_airline.columns

# Build a DataFrame with two columns: "Feature" and "Importance".
# Convenient for sorting and further analysis.
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

# Print the TOP-5 most important features.
# iterrows() iterates over rows of the DataFrame; i is the row index, row is the row itself.
for i, row in importance_df.head(5).iterrows():
    print(f"  {i+1}. {row['Feature']}: {row['Importance']:.4f} ({row['Importance']*100:.1f}%)")



In [ ]:
# Plot predictions vs. actual values
plt.figure(figsize=(10, 6))
plt.scatter(y_test_reg, lin_pred, alpha=0.6, label=f'Linear regression\n(R² = {lin_r2:.3f})')
plt.scatter(y_test_reg, rf_pred, alpha=0.6, label=f'Random forest\n(R² = {rf_r2:.3f})')

# Add the ideal line
min_val = min(y_test_reg.min(), lin_pred.min(), rf_pred.min())
max_val = max(y_test_reg.max(), lin_pred.max(), rf_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.8, label='Ideal predictions')

plt.xlabel('Actual values')
plt.ylabel('Predicted values')
plt.title('Predictions vs. Actual values')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"R² for linear regression: {lin_r2:.3f}")
print(f"R² for random forest: {rf_r2:.3f}")
print(f"RMSE for linear regression: {lin_rmse:.3f} minutes")
print(f"RMSE for random forest: {rf_rmse:.3f} minutes")
